# FLNG Technical Analysis
## Keltner Channels & Bollinger Bands

**Ticker:** FLNG (Flex LNG)  
**Analysis Period:** Last 5 days (5-minute intervals)  
**Date:** May 12, 2026

This notebook analyzes FLNG stock using:
- **Bollinger Bands** - Volatility indicator using standard deviation
- **Keltner Channels** - Volatility indicator using Average True Range (ATR)
- **Squeeze Indicator** - Identifies low volatility periods (potential breakouts)

## Setup & Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import pytz
import numpy as np
from save_to_flng_bucket import save_with_timestamp, save_csv_to_s3

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

print("[OK] Libraries imported successfully!")

## Fetch FLNG Stock Data

Fetching 5 days of minute-level stock data from Yahoo Finance.

In [ ]:
# Configuration
TICKER = 'FLNG'
DAYS_BACK = 5
INTERVAL = '5m'  # 5-minute intervals

# Fetch data from Yahoo Finance
print(f"Fetching {DAYS_BACK} days of {INTERVAL} data for {TICKER}...")

stock = yf.Ticker(TICKER)
end_date = datetime.now(pytz.timezone('US/Eastern'))
start_date = end_date - timedelta(days=DAYS_BACK)

df = stock.history(start=start_date, end=end_date, interval=INTERVAL)

# Standardize column names
df.columns = df.columns.str.title()
df.index = df.index.tz_localize(None)

print(f"[OK] Fetched {len(df)} data points")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"\nFirst 5 rows:")
df.head()

## Filter to Trading Hours

Remove pre-market and after-hours data to focus on regular trading session (9:30 AM - 4:00 PM EST).

In [ ]:
# Filter data to regular trading hours only (9:30 AM - 4:00 PM EST)
from datetime import time

# Get time component from index
times = df.index.time

# Define trading hours
market_open = time(9, 30)
market_close = time(16, 0)

# Filter for trading hours only
mask = (times >= market_open) & (times <= market_close)
df = df[mask].copy()

print(f"[OK] Filtered to trading hours only (9:30 AM - 4:00 PM EST)")
print(f"Data points: {len(df)}")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"\nFiltered DataFrame:")
df.head()

## Calculate Bollinger Bands

Bollinger Bands use a Simple Moving Average (SMA) with ±2 standard deviations.

In [ ]:
# Bollinger Bands parameters
BB_PERIOD = 20
BB_STD_DEV = 2

# Calculate middle band (SMA)
df['BB_Middle'] = df['Close'].rolling(window=BB_PERIOD).mean()

# Calculate standard deviation
rolling_std = df['Close'].rolling(window=BB_PERIOD).std()

# Calculate upper and lower bands
df['BB_Upper'] = df['BB_Middle'] + (rolling_std * BB_STD_DEV)
df['BB_Lower'] = df['BB_Middle'] - (rolling_std * BB_STD_DEV)

# Calculate bandwidth (volatility)
df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']

print(f"[OK] Calculated Bollinger Bands (period={BB_PERIOD}, std={BB_STD_DEV})")
print(f"\nCurrent BB Levels:")
print(f"  Upper: ${df['BB_Upper'].iloc[-1]:.2f}")
print(f"  Middle: ${df['BB_Middle'].iloc[-1]:.2f}")
print(f"  Lower: ${df['BB_Lower'].iloc[-1]:.2f}")
print(f"  Price: ${df['Close'].iloc[-1]:.2f}")

## Calculate Keltner Channels

Keltner Channels use an Exponential Moving Average (EMA) with ±2 × ATR (Average True Range).

In [ ]:
# Keltner Channels parameters
KC_EMA_PERIOD = 20
KC_ATR_PERIOD = 10
KC_MULTIPLIER = 2

# Calculate middle line (EMA)
df['KC_Middle'] = df['Close'].ewm(span=KC_EMA_PERIOD, adjust=False).mean()

# Calculate True Range
df['TR'] = df[['High', 'Low', 'Close']].apply(
    lambda x: max(x['High'] - x['Low'], 
                 abs(x['High'] - x['Close']), 
                 abs(x['Low'] - x['Close'])),
    axis=1
)

# Calculate Average True Range (ATR)
df['ATR'] = df['TR'].rolling(window=KC_ATR_PERIOD).mean()

# Calculate upper and lower channels
df['KC_Upper'] = df['KC_Middle'] + (df['ATR'] * KC_MULTIPLIER)
df['KC_Lower'] = df['KC_Middle'] - (df['ATR'] * KC_MULTIPLIER)

# Calculate channel width
df['KC_Width'] = df['KC_Upper'] - df['KC_Lower']

print(f"[OK] Calculated Keltner Channels (EMA={KC_EMA_PERIOD}, ATR={KC_ATR_PERIOD}, mult={KC_MULTIPLIER})")
print(f"\nCurrent KC Levels:")
print(f"  Upper: ${df['KC_Upper'].iloc[-1]:.2f}")
print(f"  Middle: ${df['KC_Middle'].iloc[-1]:.2f}")
print(f"  Lower: ${df['KC_Lower'].iloc[-1]:.2f}")
print(f"  ATR: ${df['ATR'].iloc[-1]:.2f}")

## Generate Trading Signals

Create trading signals based on price interaction with the bands and squeeze detection.

In [ ]:
# Bollinger Band signals
df['BB_Buy_Signal'] = (df['Close'] <= df['BB_Lower']).astype(int)
df['BB_Sell_Signal'] = (df['Close'] >= df['BB_Upper']).astype(int)

# Keltner Channel signals
df['KC_Buy_Signal'] = (df['Close'] <= df['KC_Lower']).astype(int)
df['KC_Sell_Signal'] = (df['Close'] >= df['KC_Upper']).astype(int)

# Squeeze indicator (BB inside KC = low volatility, potential breakout)
df['Squeeze'] = ((df['BB_Upper'] < df['KC_Upper']) & 
                 (df['BB_Lower'] > df['KC_Lower'])).astype(int)

# Combined signal
df['Signal'] = 'HOLD'
df.loc[df['BB_Buy_Signal'] == 1, 'Signal'] = 'BUY'
df.loc[df['BB_Sell_Signal'] == 1, 'Signal'] = 'SELL'

print("[OK] Generated trading signals")
print(f"\nSignal Summary:")
print(f"  Buy signals: {df['BB_Buy_Signal'].sum()}")
print(f"  Sell signals: {df['BB_Sell_Signal'].sum()}")
print(f"  Squeeze periods: {df['Squeeze'].sum()}")
print(f"  Current Signal: {df['Signal'].iloc[-1]}")

if df['Squeeze'].iloc[-1] == 1:
    print("\n  ⚠️  SQUEEZE DETECTED - Volatility compression, potential breakout coming!")

## Visualization

Create comprehensive chart showing price, Bollinger Bands, Keltner Channels, and volume.

In [ ]:
# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), 
                                 gridspec_kw={'height_ratios': [3, 1]})

# Main price chart
ax1.plot(df.index, df['Close'], label='FLNG Close Price', 
         color='black', linewidth=1.5, zorder=5)

# Bollinger Bands
ax1.plot(df.index, df['BB_Upper'], label='BB Upper', 
         color='blue', linestyle='--', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['BB_Middle'], label='BB Middle (SMA)', 
         color='blue', linestyle=':', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['BB_Lower'], label='BB Lower', 
         color='blue', linestyle='--', linewidth=1, alpha=0.7)
ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], 
                  color='blue', alpha=0.1)

# Keltner Channels
ax1.plot(df.index, df['KC_Upper'], label='KC Upper', 
         color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['KC_Middle'], label='KC Middle (EMA)', 
         color='red', linestyle=':', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['KC_Lower'], label='KC Lower', 
         color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.fill_between(df.index, df['KC_Upper'], df['KC_Lower'], 
                  color='red', alpha=0.1)

# Formatting main chart
ax1.set_title(f'FLNG - Bollinger Bands & Keltner Channels (Trading Hours Only)\n'
              f'{df.index[0].strftime("%Y-%m-%d")} to {df.index[-1].strftime("%Y-%m-%d")} | 9:30 AM - 4:00 PM EST',
              fontsize=16, fontweight='bold', pad=20)
ax1.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d\n%H:%M'))

# Volume subplot
colors = ['green' if df['Close'].iloc[i] >= df['Open'].iloc[i] 
          else 'red' for i in range(len(df))]
ax2.bar(df.index, df['Volume'], color=colors, alpha=0.5, width=0.001)
ax2.set_ylabel('Volume', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date & Time', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d\n%H:%M'))

plt.tight_layout()
plt.savefig('FLNG_Technical_Indicators.png', dpi=300, bbox_inches='tight')
plt.show()

print("[OK] Chart created and saved!")

## Save Data to S3

Save the data and analysis results to the FLNG Trading Data S3 bucket.

In [ ]:
# Save raw data to S3
raw_data_file = save_with_timestamp(df[['Open', 'High', 'Low', 'Close', 'Volume']], 
                                     'flng_raw_data', 
                                     folder='raw_data')
print(f"[OK] Raw data saved: {raw_data_file}")

# Save data with indicators
indicators_df = df[['Open', 'High', 'Low', 'Close', 'Volume',
                     'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width',
                     'KC_Upper', 'KC_Middle', 'KC_Lower', 'KC_Width',
                     'ATR']]
indicators_file = save_with_timestamp(indicators_df, 
                                       'flng_with_indicators', 
                                       folder='indicators')
print(f"[OK] Indicators saved: {indicators_file}")

# Save signals
signals_df = df[['Close', 'BB_Buy_Signal', 'BB_Sell_Signal', 
                  'KC_Buy_Signal', 'KC_Sell_Signal', 
                  'Squeeze', 'Signal']]
signals_file = save_with_timestamp(signals_df, 
                                    'flng_signals', 
                                    folder='signals')
print(f"[OK] Signals saved: {signals_file}")

## Summary Statistics

Key metrics and analysis summary for the current period.

In [ ]:
# Calculate summary statistics
current_price = df['Close'].iloc[-1]
period_high = df['High'].max()
period_low = df['Low'].min()
avg_volume = df['Volume'].mean()
price_change = df['Close'].iloc[-1] - df['Close'].iloc[0]
price_change_pct = (price_change / df['Close'].iloc[0]) * 100

print("=" * 60)
print("FLNG TECHNICAL ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nData Period:")
print(f"  Start: {df.index[0].strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  End: {df.index[-1].strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Data Points: {len(df)}")

print(f"\nPrice Statistics:")
print(f"  Current Price: ${current_price:.2f}")
print(f"  Period High: ${period_high:.2f}")
print(f"  Period Low: ${period_low:.2f}")
print(f"  Price Change: ${price_change:.2f} ({price_change_pct:+.2f}%)")

print(f"\nBollinger Bands (Current):")
print(f"  Upper: ${df['BB_Upper'].iloc[-1]:.2f}")
print(f"  Middle: ${df['BB_Middle'].iloc[-1]:.2f}")
print(f"  Lower: ${df['BB_Lower'].iloc[-1]:.2f}")
print(f"  Width: ${df['BB_Width'].iloc[-1]:.2f}")

print(f"\nKeltner Channels (Current):")
print(f"  Upper: ${df['KC_Upper'].iloc[-1]:.2f}")
print(f"  Middle: ${df['KC_Middle'].iloc[-1]:.2f}")
print(f"  Lower: ${df['KC_Lower'].iloc[-1]:.2f}")
print(f"  Width: ${df['KC_Width'].iloc[-1]:.2f}")
print(f"  ATR: ${df['ATR'].iloc[-1]:.2f}")

print(f"\nTrading Signals:")
print(f"  Total Buy Signals: {df['BB_Buy_Signal'].sum()}")
print(f"  Total Sell Signals: {df['BB_Sell_Signal'].sum()}")
print(f"  Squeeze Periods: {df['Squeeze'].sum()}")
print(f"  Current Signal: {df['Signal'].iloc[-1]}")

if df['Squeeze'].iloc[-1] == 1:
    print(f"\n  STATUS: SQUEEZE DETECTED")
    print(f"  Volatility is compressed - potential breakout coming!")
else:
    print(f"\n  STATUS: Normal volatility")

print(f"\nVolume:")
print(f"  Average: {avg_volume:,.0f}")
print(f"  Current: {df['Volume'].iloc[-1]:,.0f}")

print("=" * 60)